In [ ]:
from delta.tables import *
#from notebookutils import mssparkutils
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
#from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import col, when, from_json, date_format, lit, row_number,max, lower, regexp_extract, trim, regexp_replace
from pyspark.sql.types import StructType, StringType
from pyspark.sql.window import Window

In [ ]:
fromMonth = -16 #-1, -2,... from datenow -1 day
toMonth = -14 #-1, -2,... from datenow -1 day

## FUNCTIONS

In [ ]:
def generateArrayOFPeriod(from_Month: int, to_month: int):
    # Get today's date
    today = datetime.today()

    # Subtract one day
    yesterday = today - timedelta(days=1)
    first_day = yesterday.replace(day=1)

    periodToLoad = []
    if from_Month == to_month:
        periodDate = first_day + relativedelta(months=to_month)
        periodToLoad.append(periodDate.date())
    else:
        for i in range(from_Month, to_month+1):
            periodDate = first_day + relativedelta(months=i)
            periodToLoad.append(periodDate.date())

    return periodToLoad

## STEP 2 Load Gold:

- Filter on Fabric data only
- Remove x_ column
- Add FabricPause column

In [ ]:
periodsToLoad = generateArrayOFPeriod(fromMonth, toMonth)

In [ ]:
# Preparation of predicate for all ingested periods
date_condition = ", ".join([f"'{date.strftime('%Y-%m-%d')}'" for date in periodsToLoad])

print("Start loading Period(s) in " + date_condition)

#clean existing data in silver
if spark.catalog.tableExists("reservation_usage"):
    print("Table exists, snapshot will be clean.")
    Delete_sql_query = f"""DELETE FROM reservation_usage WHERE PeriodLoaded IN ({date_condition})"""
    spark.sql(Delete_sql_query)
    print("Clean performed")

reservation_usage_df = DeltaTable.forPath(spark,"Tables/reservation_usage_silver").toDF()
reservation_usage_df = reservation_usage_df.where(f"""PeriodLoaded IN ({date_condition})""")

In [ ]:
#Filter on Fabric RI usage only
reservation_usage_df=reservation_usage_df.filter(lower(col("InstanceId")).contains("providers/microsoft.fabric/"))\
                                        .withColumn("SubId",regexp_extract(lower(col("InstanceId")), r"subscriptions\/(.*?)\/", 1) )\
                                        .withColumn("ResourceGroupName",regexp_extract(lower(col("InstanceId")), r"resourcegroups\/(.*?)\/", 1) )\
                                        .withColumn("ResourceName",regexp_extract(lower(col("InstanceId")), r"capacities\/(.*?)$", 1) )\
                                        .withColumn("UsageDate_key",date_format("UsageDate", "yyyyMMdd").cast("int"))\
                                        .drop("InstanceId")

reservation_df = DeltaTable.forPath(spark,"Tables/reservations").toDF()
reservation_df = reservation_df.select("ReservationOrderId","ReservationOrderKey")
reservation_usage_df = reservation_usage_df.join(reservation_df,"ReservationOrderId", "leftouter") #retrieve ReservationOrderKey

subscription_df = DeltaTable.forPath(spark,"Tables/subscriptions").toDF()
subscription_df = subscription_df.select("SubId","SubscriptionKey")
reservation_usage_df = reservation_usage_df.join(subscription_df,"SubId", "leftouter") #retrieve SubscriptionKey

resources_df = DeltaTable.forPath(spark,"Tables/resources").toDF()
resources_df = resources_df.select("ResourceName","ResourceKey")
reservation_usage_df = reservation_usage_df.join(resources_df,"ResourceName", "leftouter") #retrieve ResourceKey

def null_safe_double(colname: str):
    c = trim(col(colname))
    c = regexp_replace(c, r",", ".")  # handle "1,23" if present
    return when((c == "") | (lower(c) == "null"), None).otherwise(c).cast("double")

reservation_usage_df = (
    reservation_usage_df
        .withColumn("InstanceFlexibilityRatio", null_safe_double("InstanceFlexibilityRatio"))
        .withColumn("ReservedHours",            null_safe_double("ReservedHours"))
        .withColumn("TotalReservedQuantity",    null_safe_double("TotalReservedQuantity"))
)

reservation_usage_df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable("reservation_usage")

print("End Loading")